# Yaya-125M Training
**Before running:** Runtime → Change runtime type → **T4 GPU** → Save

In [7]:
# Step 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU: NOT FOUND


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Step 2: Mount Drive (saves checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Setup
!git clone https://github.com/jaylink-coder/miss-yaya.git /content/miss-yaya 2>/dev/null || (cd /content/miss-yaya && git pull)
!pip install -q sentencepiece datasets pyyaml

In [ ]:
# Step 4: Pretrain — download OpenWebText, tokenize, and train
# This runs for ~10 hours. Checkpoints save to Google Drive automatically.
!python /content/miss-yaya/yaya-ai/scripts/colab_run.py

In [ ]:
# Step 5: SFT — instruction fine-tune the pretrained model
# Run this AFTER pretraining is complete (or after a good checkpoint is saved).
# Downloads Alpaca + Dolly data, then fine-tunes for ~3000 steps (~30 min).
!python /content/miss-yaya/yaya-ai/scripts/colab_run_sft.py

In [ ]:
# Step 6: Evaluate — see how Yaya responds to 10 standard prompts
import glob, os
sft_ckpts = sorted(glob.glob('/content/drive/MyDrive/yaya-sft-checkpoints/checkpoint-*'))
best = sft_ckpts[-1] if sft_ckpts else None
if best:
    os.system(
        f'python /content/miss-yaya/yaya-ai/scripts/eval_yaya.py '
        f'--model_config /content/miss-yaya/yaya-ai/configs/model/yaya_125m.yaml '
        f'--checkpoint {best} --chat'
    )
else:
    print('No SFT checkpoint found — run Step 5 first.')

In [ ]:
# Step 7: Chat UI — talk to Yaya in your browser
# Opens a public Gradio link. Works even on free Colab.
import glob
sft_ckpts = sorted(glob.glob('/content/drive/MyDrive/yaya-sft-checkpoints/checkpoint-*'))
best = sft_ckpts[-1] if sft_ckpts else None
if best:
    import os
    os.system(
        f'python /content/miss-yaya/yaya-ai/scripts/web_ui.py '
        f'--model_config /content/miss-yaya/yaya-ai/configs/model/yaya_125m.yaml '
        f'--checkpoint {best} --share'
    )
else:
    print('No SFT checkpoint found — run Step 5 first.')